# Hidden Markov Model — Parts 1 & 2
## Viterbi, Forward, Backward, and Posterior Decoding

---

This notebook covers the implementation of a Hidden Markov Model (HMM) across two parts:

- **Part 1** — The Viterbi Algorithm: finding the most likely hidden state sequence
- **Part 2** — The Forward Algorithm, Backward Algorithm, and Posterior Decoding

All probability computations are performed in **log space** to avoid numerical underflow when multiplying many small probabilities together.

---
## Imports

In [16]:
import numpy as np
from numbers import Number
from collections.abc import Iterable
from math import log

---
## The `State` Class

Each `State` represents a single hidden state in the HMM. It stores:
- Its **emission probabilities** — how likely each observation symbol is from this state
- Its **transition probabilities** — how likely it is to move to each other state

Emission probabilities must sum to 1. If a new emission is added that pushes the total over 1, all probabilities are rescaled to maintain relative proportions.

In [17]:
class State:
    """Hidden state for HMM"""
    def __init__(self, name:str, emissions: list, probabilities: list[Number], transitions:dict[str:float]):
        self.name = name
        self.emissions = set(emissions)
        self.emission_probs = dict(zip(emissions, probabilities))
        self.transitions = transitions
        self.total_emission_prob = sum(probabilities)
        if self.total_emission_prob != 1:
            raise ValueError("Emission probabilities do not sum to 1")
        
    def __repr__(self):
        return self.name

    def add_emission(self, emission, probability: Number):
        """
        Add emission:probability pair to State emissions dictionary
        """
        self.emissions.add(emission)
        self.emission_probs[emission] = probability
        self.total_emission_prob += probability

        if self.total_emission_prob > 1: 
            print(f"\nWARNING: sum of State: {self.name} emission probabilities has exceeded 1.\n"
                  f"Refactoring to maintain relative probabilities with sum of 1\n")
            for emit in self.emissions:
                self.emission_probs[emit] = self.emission_probs[emit] / self.total_emission_prob

---
## The `HMM` Class

The `HMM` class ties everything together. It holds:
- A list of `State` objects
- A **transition matrix** (`t_mat`) built automatically from the states' transition dictionaries, where `t_mat[i][j]` is the probability of going from state `i` to state `j`
- **Initial probabilities** (`betas`) — the probability of starting in each state (equivalent to π)
- The full set of possible **emissions**

It also contains all four algorithm implementations as methods.

In [18]:
class HMM():
    def __init__(self, name: str, betas: dict[State:float], emissions: set, states: list[State]):
        self.name = name
        self.states = states
        self.t_mat = self.build_transition_mat_from_states()
        self.emissions = set(emissions)
        self.betas = betas

    def __repr__(self):
        return(f"HMM Class\n"
               f"Name: {self.name}\n"
               f"Emissions: {self.emissions}\n"
               f"Model States: {self.states}\n"
               f"Transition Matrix:\n{self.t_mat}\n"
               f"Betas: {self.betas}")
    
    def build_transition_mat_from_states(self):
        """
        Build a transition matrix from the transition probabilities of states in HMM.
        t_mat[i][j] = probability of transitioning from state i to state j.
        """
        t_mat = [[] for state in self.states]
        for row, state in enumerate(self.states):
            for trans in self.states:
                t_mat[row].append(state.transitions[trans.name])
        return np.array(t_mat)

    def add_state(self, name:str = None, emissions: list = [], probabilities: list = [], state:State = None):
        """
        Add a State to the HMM either by providing a State object or arguments to create one.
        """
        if state is None:
            if name is None:
                raise ValueError("No name given to create new State object from arguments")
            state = State(name=name, emissions=emissions, probabilities=probabilities)

        self.states.append(state)

        for emission in state.emissions:
            if not emission in self.emissions:
                self.emissions.add(emission)

        for emission in self.emissions:
            for state in self.states:
                if not emission in state.emissions:
                    state.add_emission(emission=emission, probability=0)

---
## Part 1 — Viterbi Algorithm

### What it answers
> *What is the single most likely sequence of hidden states given the observations?*

### How it works
The Viterbi algorithm uses **dynamic programming** to find the best path through the HMM:

1. **Initialize** — compute the log probability of being in each state at position 0 using initial probabilities and the first emission
2. **Recurse left to right** — at each position, for each state, take the **max** (not sum) over all incoming paths
3. **Traceback** — follow the saved best-previous-state pointers from end to start to recover the full path

### Pseudocode
```
VITERBI(observations):

    # Initialize first column
    for each state s:
        vit[s][0]       = log(betas[s]) + log(emission(s, obs[0]))
        traceback[s][0] = STOP signal

    # Fill remaining columns left to right
    for each obs index t from 1 to T-1:
        for each state s:
            options      = [ vit[s'][t-1] + log(t_mat[s'→s]) + log(emission(s, obs[t])) for each s' ]
            vit[s][t]       = MAX of options
            traceback[s][t] = ARGMAX of options

    # Traceback from best final state
    best_final_state = ARGMAX of vit[:, T-1]
    follow traceback pointers from end to start
    return reversed state path
```

In [19]:
def _get_prev_state_options(self, obs:int, observations: Iterable, mat:np.ndarray, mat_row:int) -> list:
    """
    Shared helper for Viterbi and Forward.
    For each previous state, computes:
        mat[prev_state][obs-1] + log(t_mat[prev_state -> current]) + log(emission(current, obs[t]))
    """
    options = [state_row[obs-1] + log(self.t_mat[trans_state][mat_row]) + log(self.states[mat_row].emission_probs[observations[obs]])
                for trans_state, state_row in enumerate(mat)]
    return options

def viterbi(self, observations: Iterable) -> list[State]:
    """
    Calculate most likely state sequence given observations.
    Uses MAX at each step and traces back the best path.
    """
    vit = np.zeros((len(self.states), len(observations)))
    traceback = np.zeros((len(self.states), len(observations)), dtype=int)

    for row in traceback:
        row[0] = len(self.states)  # stop signal

    for state, row in enumerate(vit):
        row[0] = log(self.betas[self.states[state]]) + log(self.states[state].emission_probs[observations[0]])

    for obs in range(1, len(observations)):
        for state, row in enumerate(vit):
            options = _get_prev_state_options(self, obs=obs, observations=observations, mat=vit, mat_row=state)
            row[obs] = max(options)
            traceback[state][obs] = np.argmax(options)

    end_states = [row[-1] for row in vit]
    max_path = np.argmax(end_states)

    index = -1
    trace = max_path
    state_path = [max_path]
    while traceback[trace][index] != len(self.states):
        trace = traceback[trace][index]
        state_path.append(trace)
        index -= 1

    for ind, code in enumerate(state_path):
        state_path[ind] = self.states[code]

    return list(reversed(state_path))

HMM._get_prev_state_options = _get_prev_state_options
HMM.viterbi = viterbi

---
## Part 2 — Forward Algorithm

### What it answers
> *What is the total probability of seeing this observation sequence?*

### How it works
The Forward algorithm is nearly identical to Viterbi, but instead of taking the **max** over incoming paths, it takes the **sum** — accounting for every possible way to have arrived at each state.

In log space, summing is done with `np.logaddexp.reduce()` instead of `max()`.

### Pseudocode
```
FORWARD(observations):

    # Initialize first column
    for each state s:
        forward_mat[s][0] = log(betas[s]) + log(emission(s, obs[0]))

    # Fill remaining columns left to right
    for each obs index t from 1 to T-1:
        for each state s:
            options          = [ forward_mat[s'][t-1] + log(t_mat[s'→s]) + log(emission(s, obs[t])) for each s' ]
            forward_mat[s][t] = SUM of options  (logaddexp in log space)

    return forward_mat
```

In [20]:
def forward(self, observations: Iterable) -> np.ndarray:
    """
    Compute the forward matrix.
    forward_mat[s][t] = log P(obs[0..t], state=s at t)
    """
    forward_mat = np.ndarray((len(self.states), len(observations)))

    # Initialize first column
    for state, row in enumerate(forward_mat):
        beta = self.betas[self.states[state]]
        emission = self.states[state].emission_probs[observations[0]]
        row[0] = log(beta) + log(emission)

    # Fill left to right — SUM over all previous states (not max)
    for obs_ind in range(1, len(observations)):
        for state, row in enumerate(forward_mat):
            options = self._get_prev_state_options(obs=obs_ind, observations=observations, mat=forward_mat, mat_row=state)
            row[obs_ind] = np.logaddexp.reduce(options)

    return forward_mat

HMM.forward = forward

---
## Part 2 — Backward Algorithm

### What it answers
> *Given that I'm in state `s` at time `t`, what is the probability of seeing the rest of the observations from `t+1` onward?*

### How it works
The Backward algorithm is the mirror image of Forward — it sweeps **right to left**.

- **Initialize** the last column to `0` (which is `log(1)`) — there are no future observations at the end, so the probability is 1
- **Recurse right to left** — for each state at time `t`, sum over all next states using transition, next emission, and the future beta value
- Note: the result is stored one column to the **left** of the current index (`obs_ind - 1`)

### Pseudocode
```
BACKWARD(observations):

    # Initialize last column to 0 (= log(1))
    for each state s:
        backward_mat[s][T-1] = 0

    # Fill remaining columns right to left
    for each obs index t from T-1 down to 1:
        for each state s:
            options = [ log(t_mat[s→s']) + log(emission(s', obs[t])) + backward_mat[s'][t]
                        for each next state s' ]
            backward_mat[s][t-1] = SUM of options  (logaddexp in log space)

    return backward_mat
```

In [21]:
def _get_future_options(self, obs_ind:int, observations:Iterable, mat:np.ndarray, mat_row:int) -> list:
    """
    Helper for Backward.
    For each next state, computes:
        log(t_mat[current -> next]) + log(emission(next, obs[t])) + mat[next][t]
    """
    options = [log(self.t_mat[mat_row][state]) + log(self.states[state].emission_probs[observations[obs_ind]]) + mat[state][obs_ind]
               for state, row in enumerate(mat)]
    return options

def backward(self, observations: Iterable) -> np.ndarray:
    """
    Compute the backward matrix.
    backward_mat[s][t] = log P(obs[t+1..T] | state=s at t)
    """
    backward_mat = np.ndarray((len(self.states), len(observations)))

    # Initialize last column to 0 = log(1)
    for row in backward_mat:
        row[-1] = 0

    # Fill right to left — store result one column to the left
    for obs_ind in range(len(observations)-1, 0, -1):
        for state, row in enumerate(backward_mat):
            options = self._get_future_options(observations=observations, obs_ind=obs_ind, mat=backward_mat, mat_row=state)
            row[obs_ind-1] = np.logaddexp.reduce(options)

    return backward_mat

HMM._get_future_options = _get_future_options
HMM.backward = backward

---
## Part 2 — Posterior Decoding

### What it answers
> *At each position in the sequence, which hidden state was most likely — considering ALL observations (both past and future)?*

### How it works
Posterior Decoding combines the Forward and Backward matrices to compute **gamma** — the posterior probability of being in each state at each time step:

$$\gamma(t, s) = \frac{\alpha(t,s) \cdot \beta(t,s)}{P(\text{observations})}$$

In log space: `gamma = forward_mat + backward_mat - log_prob_obs`

The most likely state at each position is then simply `argmax` over gamma.

### Key difference from Viterbi
| | Viterbi | Posterior Decoding |
|---|---|---|
| Considers | Whole path jointly | Each position independently |
| Uses | Forward only + traceback | Forward + Backward |
| Guarantees | A valid consistent path | Locally optimal at each step |

### Pseudocode
```
POSTERIOR_DECODING(observations):

    forward_mat  = FORWARD(observations)
    backward_mat = BACKWARD(observations)

    # Total log probability of observations (sum last column of forward in log space)
    log_prob_obs = logaddexp.reduce(forward_mat[:, -1])

    # Gamma: log P(state s at time t | all observations)
    gamma = forward_mat + backward_mat - log_prob_obs

    # At each time step, pick the state with the highest gamma
    best_state_indices = argmax(gamma, axis=0)

    return [ states[i] for i in best_state_indices ]
```

In [22]:
def posterior_decoding(self, observations: Iterable) -> list[State]:
    """
    Compute the most likely state at each position using all observations.
    Combines Forward and Backward via gamma = forward + backward - log_prob_obs.
    """
    forward_mat  = self.forward(observations)
    backward_mat = self.backward(observations)

    # log P(observations) = sum over all states at the final time step
    log_prob_obs = np.logaddexp.reduce(forward_mat[:, -1])

    # Gamma in log space
    gamma = forward_mat + backward_mat - log_prob_obs

    # Pick the best state at each position
    best_state_indices = np.argmax(gamma, axis=0)

    return [self.states[i] for i in best_state_indices]

HMM.posterior_decoding = posterior_decoding

---
## Example: Setting Up the HMM

We define two hidden states (`my_state` and `my_state2`) each with emissions over the alphabet `{A, B, C}` and run all four algorithms on the observation sequence `"ABACB"`.

In [23]:
# Define states
my_state = State(
    name="my_state",
    emissions=["A", "B", "C"],
    probabilities=[0.3, 0.2, 0.5],
    transitions={"my_state": 0.7, "my_state2": 0.3}
)

my_state2 = State(
    name="my_state2",
    emissions=["A", "B", "C"],
    probabilities=[0.2, 0.7, 0.1],
    transitions={"my_state2": 0.9, "my_state": 0.1}
)

# Build HMM
my_HMM = HMM(
    name="My_HMM",
    betas={my_state: 0.5, my_state2: 0.5},
    emissions={"A", "B", "C"},
    states=[my_state, my_state2]
)

print(my_HMM)

HMM Class
Name: My_HMM
Emissions: {'A', 'C', 'B'}
Model States: [my_state, my_state2]
Transition Matrix:
[[0.7 0.3]
 [0.1 0.9]]
Betas: {my_state: 0.5, my_state2: 0.5}


---
## Part 1 — Running Viterbi

In [24]:
observations = "ABACB"

viterbi_path = my_HMM.viterbi(observations=observations)
print("Observation sequence : ", list(observations))
print("Viterbi state path   : ", viterbi_path)

Observation sequence :  ['A', 'B', 'A', 'C', 'B']
Viterbi state path   :  [my_state2, my_state2, my_state2, my_state2, my_state2]


---
## Part 2 — Running Forward & Backward

In [25]:
forward_mat  = my_HMM.forward(observations=observations)
backward_mat = my_HMM.backward(observations=observations)

print("Forward matrix (log space):")
print(forward_mat)

print("\nBackward matrix (log space):")
print(backward_mat)

# Total log probability of the observation sequence
log_prob = np.logaddexp(forward_mat[0][-1], forward_mat[1][-1])
print(f"\nlog P(observations) from Forward : {log_prob:.4f}")
print(f"P(observations)                  : {np.exp(log_prob):.6f}")

Forward matrix (log space):
[[-1.89711998 -3.77226106 -4.87109077 -5.62619663 -7.52021504]
 [-2.30258509 -2.35915544 -3.99594824 -6.27380093 -6.2429798 ]]

Backward matrix (log space):
[[-4.75825144 -3.37028028 -1.95192822 -1.04982212  0.        ]
 [-4.42369899 -4.02072242 -2.57702194 -0.43078292  0.        ]]

log P(observations) from Forward : -5.9971
P(observations)                  : 0.002486


---
## Part 2 — Running Posterior Decoding

In [26]:
posterior_path = my_HMM.posterior_decoding(observations=observations)

print("Observation sequence      : ", list(observations))
print("Viterbi path              : ", viterbi_path)
print("Posterior Decoding path   : ", posterior_path)
print()
print("Note: Viterbi finds the single best joint path.")
print("Posterior Decoding picks the best state at each position independently.")
print("They may agree or differ depending on the transition structure.")

Observation sequence      :  ['A', 'B', 'A', 'C', 'B']
Viterbi path              :  [my_state2, my_state2, my_state2, my_state2, my_state2]
Posterior Decoding path   :  [my_state, my_state2, my_state2, my_state, my_state2]

Note: Viterbi finds the single best joint path.
Posterior Decoding picks the best state at each position independently.
They may agree or differ depending on the transition structure.
